## First step

In [1]:
!python unzip.py

17 archive(s) found. Starting extraction...

Skipping 'roberta_fine_tunned.zip', already extracted at 'style_branch/roberta_fine_tunned/'.
Skipping 'roberta_fake_news.zip', already extracted at 'style_branch/roberta_fake_news/'.
Skipping 'bloc_C_test_final_WITH_PROB.csv.zip', already extracted at 'data/bloc_C_test_final_WITH_PROB.csv/'.
Skipping 'complete_train.csv.zip', already extracted at 'data/complete_train.csv/'.
Skipping 'bloc_A_roberta_train.csv.zip', already extracted at 'data/bloc_A_roberta_train.csv/'.
Skipping 'bloc_C_test_final.csv.zip', already extracted at 'data/bloc_C_test_final.csv/'.
Skipping 'bloc_B_xgb_train.csv.zip', already extracted at 'data/bloc_B_xgb_train.csv/'.
Skipping 'bloc_B_xgb_train_WITH_PROB.csv.zip', already extracted at 'data/bloc_B_xgb_train_WITH_PROB.csv/'.
Skipping 'test.csv.zip', already extracted at 'data/dataset_kaggle/test.csv/'.
Skipping 'train.csv.zip', already extracted at 'data/dataset_kaggle/train.csv/'.
Skipping 'base_valid.zip', already 

## Style based detection

### 1. Data preparation

If it has already been run, there is no need to run it again (but you can if you have 40 spare minutes)

In [6]:
%cd data/fake_news_detection_UoVictoria
!python merge_datasets.py
%cd ../..

%cd style_branch
!python feature_extraction.py

!python print_features.py
%cd .. # important ++ if we want to execute many times a cell

/Users/ethan/Documents/GitHub/Detection_fake_news/data/fake_news_detection_UoVictoria
Loading Fake.csv and True.csv files...
Dataset merged successfully! Total rows: 44898
Fake news: 23481 | True news: 21417

DONE! The dataset is ready and saved as: dataset_ready.csv
/Users/ethan/Documents/GitHub/Detection_fake_news
/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Loading data from ../data/fake_news_detection_UoVictoria/dataset_ready.csv...

Initializing NLP engine (spaCy + TextBlob/VADER)...
Loading linguistic models...
Extracting Style Metrics: 100%|███████████| 44898/44898 [40:51<00:00, 18.32it/s]

DONE! Features have been saved to: ../data/complete_train.csv
NEWS ANALYSIS:

Raw text:
ALERT! You must absolutely read this: https://www.google.com. The government is hiding 10,000 terrible and monstrous things! Wake up immediately @user! #news 😡

Loading linguistic models...
Normalized text:
ALERT! You must absolutely read this: [URL] The government is hiding [NB] terrible

### 2. Fine tuning RoBERTa
Same as the previous cell, you don't need to run it, it can be very very very long because of your GPU (if you have it)
(MacBook Pro M4 with 20 GPU core (48 Go unified memory) -> 41 minutes)

In [ ]:
%cd style_branch
!python split_data.py

!python fine_tunning.py

%run test_fine_tuned.py
%cd ..

[Errno 2] No such file or directory: 'style_branch'
/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Loading the complete dataset (Style + Text)...

Data distribution:
Block A (RoBERTa Train) : 26938 rows
Block B (XGBoost Train) : 8980 rows
Block C (Final Test)    : 8980 rows

Sanity Check - Class Distribution (Fake=1, True=0):
Block A: 
label
1    0.523
0    0.477
Name: proportion, dtype: float64
Block B: 
label
1    0.523
0    0.477
Name: proportion, dtype: float64
Block C: 
label
1    0.523
0    0.477
Name: proportion, dtype: float64

Splitting complete and files saved!
Apple Silicon (MPS) detected! Training will be hardware-accelerated.
Loading CSV files...
Word tokenization in progress... This may take a minute.
Loading weights: 100%|██████████████████████| 101/101 [00:00<00:00, 8461.66it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.w

### 3. Random Forest x XGBoost competition
If warnings appears about killing child process it is because you are on MacOs and it auto kill child process and the Python processus find any child process already kild by native MacOs processsus so it's just a warning and no action required. Others os aren't concerned.

In [8]:
%cd style_branch
!python model_comp.py

/Users/ethan/Documents/GitHub/Detection_fake_news/style_branch
Loading data (Style + RoBERTa Semantics)...

Configuration: 15 combinations tested per model (Cross-Validation x5).
Optimizing Random Forest...

Training Random Forest: 100%|███████████████████| 75/75 [00:10<00:00,  7.32it/s]
Finished in 0.17 min. Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 20, 'class_weight': 'balanced'}
Optimizing XGBoost...

Training XGBoost: 100%|█████████████████████████| 75/75 [00:01<00:00, 72.59it/s]
Finished in 0.02 min. Best parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

Model                | Accuracy             | ROC-AUC (Quality)    | F1 Score             | Log Loss             |
------------------------------------------------------------------------------------------------------------------
Random Forest        |               99.99% |              100.00% |           

In [ ]:
# it's better to use in terminal

# Tests
# BREAKING NEWS!!! 🚨 Government officials are secretly hiding ALIEN technology under the White House! The mainstream media is LYING to you!!! WAKE UP! #Truth #Conspiracy
# WASHINGTON (Reuters) - The Federal Reserve announced a 0.5% increase in interest rates on Tuesday, aiming to stabilize inflation across the country.
%run inference_pipeline.py

Initializing Detector (Loading into RAM...)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Detector is ready and loaded in memory!

 Type 'q' to quit.
Loading linguistic models...
FAKE NEWS (Confidence: 99.7%)
Input text: BREAKING NEWS!!! 🚨 Government officials are secretly hiding ALIEN technology under the White House! The mainstream media is LYING to you!!! WAKE UP! #Truth #Conspiracy
Loading linguistic models...
REAL NEWS (Confidence: 80.2%)
Input text: WASHINGTON (Reuters) - The Federal Reserve announced a 0.5% increase in interest rates on Tuesday, aiming to stabilize inflation across the country. %run inference_pipeline.py
